In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("city_day.csv", parse_dates=["Date"])
stations = pd.read_csv("stations.csv")
df = df.sort_values(["City","Date"]).reset_index(drop=True)

city_state = stations[["City","State"]].drop_duplicates()
df = df.merge(city_state, on="City", how="left")

print("Shape:", df.shape)
df.head()

Shape: (29531, 17)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,State
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN,Gujarat
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN,Gujarat
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN,Gujarat
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN,Gujarat
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN,Gujarat


In [3]:
#  Fill missing values by interpolation
pollutants = ["PM2.5","PM10","NO","NO2","NOx","NH3","CO","SO2","O3","Benzene","Toluene","Xylene","AQI"]

before = df[pollutants].isnull().sum().sum()

df[pollutants] = (
    df.groupby("City")[pollutants]
    .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

df = df.dropna(subset=["AQI"]).reset_index(drop=True)

after = df[pollutants].isnull().sum().sum()
print(f"Missing values before: {before}")
print(f"Missing values after : {after}")
print(f"Rows after cleaning  : {len(df)}")

Missing values before: 83807
Missing values after : 25138
Rows after cleaning  : 29531


In [4]:
# Date & season features
df["month"]       = df["Date"].dt.month
df["day_of_week"] = df["Date"].dt.dayofweek
df["year"]        = df["Date"].dt.year
df["quarter"]     = df["Date"].dt.quarter

df["season"] = df["month"].map({
    12: 0, 1: 0, 2: 0,      # Winter
    3: 1,  4: 1, 5: 1,      # Spring
    6: 2,  7: 2, 8: 2, 9:2, # Monsoon
    10: 3, 11: 3             # Autumn
})

print("Date features added:", ["month","day_of_week","year","quarter","season"])
df[["City","Date","month","season","AQI"]].head(8)

Date features added: ['month', 'day_of_week', 'year', 'quarter', 'season']


,City,Date,month,season,AQI
0,Ahmedabad,2015-01-01,1,0,209.0
1,Ahmedabad,2015-01-02,1,0,209.0
2,Ahmedabad,2015-01-03,1,0,209.0
3,Ahmedabad,2015-01-04,1,0,209.0
4,Ahmedabad,2015-01-05,1,0,209.0
5,Ahmedabad,2015-01-06,1,0,209.0
6,Ahmedabad,2015-01-07,1,0,209.0
7,Ahmedabad,2015-01-08,1,0,209.0


In [5]:
#  Lag features (most important for time series ML)
# Previous day AQI values — tells the model recent trend
for lag in [1, 3, 7]:
    df[f"AQI_lag_{lag}"] = df.groupby("City")["AQI"].shift(lag)

# Rolling averages — smoothed recent history
df["AQI_rolling_3"]  = df.groupby("City")["AQI"].transform(
    lambda x: x.rolling(3,  min_periods=1).mean().shift(1))
df["AQI_rolling_7"]  = df.groupby("City")["AQI"].transform(
    lambda x: x.rolling(7,  min_periods=1).mean().shift(1))
df["AQI_rolling_30"] = df.groupby("City")["AQI"].transform(
    lambda x: x.rolling(30, min_periods=1).mean().shift(1))

df = df.dropna().reset_index(drop=True)

print("Shape after lag features:", df.shape)
df[["City","Date","AQI","AQI_lag_1","AQI_lag_7","AQI_rolling_7"]].head(8)

Shape after lag features: (11726, 28)


,City,Date,AQI,AQI_lag_1,AQI_lag_7,AQI_rolling_7
0,Amaravati,2017-12-01,191.0,165.0,184.0,184.142857
1,Amaravati,2017-12-02,191.0,191.0,184.0,185.142857
2,Amaravati,2017-12-03,227.0,191.0,197.0,186.142857
3,Amaravati,2017-12-04,168.0,227.0,198.0,190.428571
4,Amaravati,2017-12-05,198.0,168.0,188.0,186.142857
5,Amaravati,2017-12-06,201.0,198.0,173.0,187.571429
6,Amaravati,2017-12-07,252.0,201.0,165.0,191.571429
7,Amaravati,2017-12-08,310.0,252.0,191.0,204.000000


In [6]:
#  Define features and target
feature_cols = [
    # Pollutants
    "PM2.5", "PM10", "NO2", "CO", "O3", "SO2", "Benzene",
    # Date features
    "month", "day_of_week", "season", "quarter",
    # Lag features
    "AQI_lag_1", "AQI_lag_3", "AQI_lag_7",
    "AQI_rolling_3", "AQI_rolling_7", "AQI_rolling_30"
]

X = df[feature_cols]
y = df["AQI"]

print("Features:", len(feature_cols))
print("Dataset size:", X.shape)

# Check correlation of new features with AQI
corr = df[feature_cols + ["AQI"]].corr()["AQI"].sort_values(ascending=False)
print("\nTop correlations with AQI:")
print(corr.head(10).round(3))

Features: 17
Dataset size: (11726, 17)

Top correlations with AQI:
AQI               1.000
AQI_lag_1         0.921
AQI_rolling_3     0.893
PM2.5             0.892
AQI_rolling_7     0.874
AQI_rolling_30    0.836
AQI_lag_3         0.815
AQI_lag_7         0.767
PM10              0.746
NO2               0.436
Name: AQI, dtype: float64


In [8]:
#  Train/test split by date
SPLIT_DATE = "2020-01-01"

train_df = df[df["Date"] < SPLIT_DATE].copy()
test_df  = df[df["Date"] >= SPLIT_DATE].copy()

X_train = train_df[feature_cols]
y_train = train_df["AQI"]
X_test  = test_df[feature_cols]
y_test  = test_df["AQI"]

print(f"Train: {X_train.shape}  rows: {train_df['Date'].min().date()} → {train_df['Date'].max().date()}")
print(f"Test : {X_test.shape}   rows: {test_df['Date'].min().date()} → {test_df['Date'].max().date()}")

# Save clean data
df.to_csv("city_day_clean.csv", index=False)
print("\nSaved: city_day_clean.csv")

Train: (9911, 17)  rows: 2015-01-08 → 2019-12-31
Test : (1815, 17)   rows: 2020-01-01 → 2020-07-01

Saved: city_day_clean.csv
